DSCI 552 Homework 3
===================

- Name: Umaeshwer Shankar
- GitHub Username: umaeshwer
- USD ID: 7601-5514-88

# Importing Libraries

In [81]:
import os
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import bootstrapped.bootstrap as bs
import bootstrapped.stats_functions as bs_stats

## 1. Time Series Classification Part 1: Feature Creation/Extraction

(a) Download the AReM data. The dataset contains 7 folders that represent seven types of activities. In
each folder, there are multiple files each of which represents an instant of a human
performing an activity.1 Each file containis 6 time series collected from activities
of the same person, which are called avg rss12, var rss12, avg rss13, var rss13,
vg rss23, and ar rss23. There are 88 instances in the dataset, each of which con-
tains 6 time series and each time series has 480 consecutive values.

In [23]:
root_path = '../data/AReM'

folders = [f for f in os.listdir(root_path) if os.path.isdir(os.path.join(root_path, f))]
print(f"Folder names: {folders}")

Folder names: ['bending1', 'walking', 'bending2', 'standing', 'sitting', 'lying', 'cycling']


(b) Keep datasets 1 and 2 in folders bending1 and bending 2, as well as datasets 1,
2, and 3 in other folders as test data and other datasets as train data.

In [22]:
train_files, test_files = [], []

def load_files(folder_path):
    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.endswith('.csv'):
                if root.split('/')[3] in ['bending1', 'bending2'] and file in ['dataset1.csv', 'dataset2.csv']:
                    test_files.append(root + '/' + file)
                elif root.split('/')[3] not in ['bending1', 'bending2'] and file in ['dataset1.csv', 'dataset2.csv', 'dataset3.csv']:
                    test_files.append(root + '/' + file)
                else:
                    train_files.append(root + '/' + file)
    return train_files, test_files

train_files, test_files = load_files(root_path)

print(f"Train files: {train_files}")
print(f"Test files: {test_files}")

Train files: ['../data/AReM/bending1/dataset7.csv', '../data/AReM/bending1/dataset6.csv', '../data/AReM/bending1/dataset4.csv', '../data/AReM/bending1/dataset5.csv', '../data/AReM/bending1/dataset3.csv', '../data/AReM/walking/dataset7.csv', '../data/AReM/walking/dataset6.csv', '../data/AReM/walking/dataset4.csv', '../data/AReM/walking/dataset5.csv', '../data/AReM/walking/dataset10.csv', '../data/AReM/walking/dataset11.csv', '../data/AReM/walking/dataset13.csv', '../data/AReM/walking/dataset12.csv', '../data/AReM/walking/dataset15.csv', '../data/AReM/walking/dataset14.csv', '../data/AReM/walking/dataset8.csv', '../data/AReM/walking/dataset9.csv', '../data/AReM/bending2/dataset6.csv', '../data/AReM/bending2/dataset4.csv', '../data/AReM/bending2/dataset5.csv', '../data/AReM/bending2/dataset3.csv', '../data/AReM/standing/dataset7.csv', '../data/AReM/standing/dataset6.csv', '../data/AReM/standing/dataset4.csv', '../data/AReM/standing/dataset5.csv', '../data/AReM/standing/dataset10.csv', '..

(c) Feature Extraction

Classification of time series usually needs extracting features from them. In this
problem, we focus on time-domain features.

i. Research what types of time-domain features are usually used in time series
classification and list them (examples are minimum, maximum, mean, etc).

Answer:

- Mean
- Median
- Minimum
- Maximum
- Standard Deviation
- 1st quartile
- 3rd quartile
- Interquartile range
- Lag

ii. Extract the time-domain features minimum, maximum, mean, median, standard deviation, first quartile, and third quartile for all of the 6 time series
in each instance. You are free to normalize/standardize features or use them
directly.

In [71]:
column_names = ['# Columns: time','avg_rss12','var_rss12','avg_rss13','var_rss13','avg_rss23','var_rss23']
time_series_columns = column_names[1:]
feature_columns = ['mean', 'std', 'min', '1st_quart', 'median', '3rd_quart', 'max']
def load_data(file_list):
    df_dict = {}
    for file in file_list:
        try:
            df = pd.read_csv(file, skiprows=5, header=None,on_bad_lines='skip')

            df.columns = column_names
            df = df.describe().drop('count').drop(columns='# Columns: time').T
            df.columns = feature_columns
            df = df.to_dict()
            for col in time_series_columns:
                for feat in feature_columns:
                    if col + '_' + feat in df_dict.keys():
                        df_dict[col + '_' + feat].append(df[feat][col])
                    else:
                        df_dict[col + '_' + feat] = [df[feat][col]]
        except Exception as e:
            print(f"Error reading file {file}: {e}")
            print("Trying with tab separator...")
            df = pd.read_csv(file, skiprows=5, sep='\\s', header=None,on_bad_lines='skip')
        
            df.columns = column_names
            df = df.describe().drop('count').drop(columns='# Columns: time').T
            df.columns = feature_columns
            df = df.to_dict()
            for col in time_series_columns:
                for feat in feature_columns:
                    if col + '_' + feat in df_dict.keys():
                        df_dict[col + '_' + feat].append(df[feat][col])
                    else:
                        df_dict[col + '_' + feat] = [df[feat][col]]

    return pd.DataFrame(df_dict)


In [74]:
"""
Errors encountered:

1. ../data/AReM/bending2/dataset4.csv: Different delimiter
2. Skipping bad lines in some files due to inconsistent formatting
"""

print("Loading training data...")
train_df = load_data(train_files)

display(train_df)

Loading training data...
Error reading file ../data/AReM/bending2/dataset4.csv: Length mismatch: Expected axis has 1 elements, new values have 7 elements
Trying with tab separator...


/var/folders/5c/3qs4gb0n2jjb1nf6m8w53dvh0000gn/T/ipykernel_43080/1450568240.py:23: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv(file, skiprows=5, sep='\\s', header=None,on_bad_lines='skip')


,avg_rss12_mean,avg_rss12_std,avg_rss12_min,avg_rss12_1st_quart,avg_rss12_median,avg_rss12_3rd_quart,avg_rss12_max,var_rss12_mean,var_rss12_std,var_rss12_min,...,avg_rss23_median,avg_rss23_3rd_quart,avg_rss23_max,var_rss23_mean,var_rss23_std,var_rss23_min,var_rss23_1st_quart,var_rss23_median,var_rss23_3rd_quart,var_rss23_max
0,43.969125,1.618364,36.25,43.310,44.50,44.67,48.00,0.413125,0.263111,0.0,...,21.67,23.7500,30.75,0.555312,0.487826,0.0,0.0000,0.490,0.830,2.96
1,43.454958,1.386098,37.00,42.500,43.25,45.00,48.00,0.378083,0.315566,0.0,...,23.50,24.0000,33.50,0.679646,0.622534,0.0,0.4300,0.500,0.870,5.26
2,42.179812,3.670666,33.00,39.150,43.50,45.00,47.75,0.696042,0.630860,0.0,...,35.00,36.3300,38.67,0.613521,0.524317,0.0,0.0000,0.500,1.000,2.18
3,41.678063,2.243490,33.00,41.330,41.75,42.75,45.75,0.535979,0.405469,0.0,...,30.00,31.2500,37.50,0.383292,0.389164,0.0,0.0000,0.430,0.500,1.79
4,43.954500,1.558835,35.00,43.000,44.33,45.00,47.40,0.426250,0.338690,0.0,...,36.00,36.5000,38.50,0.493292,0.513506,0.0,0.0000,0.430,0.940,1.79
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64,35.752354,4.614802,18.50,33.000,36.00,39.33,44.25,3.328104,2.140576,0.0,...,16.25,18.0625,24.33,3.069667,1.748326,0.0,1.7975,2.770,4.060,9.39
65,37.177042,3.581301,24.25,34.500,36.25,40.25,45.00,2.374208,1.601799,0.0,...,20.00,21.7500,25.50,2.921729,1.852600,0.0,1.5000,2.500,3.900,9.34
66,36.248768,3.824632,23.33,33.415,36.75,39.25,43.50,2.737307,2.093999,0.0,...,18.50,21.0000,27.00,3.532463,1.965267,0.0,2.1700,3.110,4.625,11.15
67,36.957458,3.434863,26.25,34.500,36.29,40.25,44.25,2.420083,1.724901,0.0,...,16.33,18.2500,22.25,2.934625,1.631380,0.0,1.6600,2.525,4.030,8.34


In [75]:
print("Loading test data...")
test_df = load_data(test_files)

display(test_df)

Loading test data...


,avg_rss12_mean,avg_rss12_std,avg_rss12_min,avg_rss12_1st_quart,avg_rss12_median,avg_rss12_3rd_quart,avg_rss12_max,var_rss12_mean,var_rss12_std,var_rss12_min,...,avg_rss23_median,avg_rss23_3rd_quart,avg_rss23_max,var_rss23_mean,var_rss23_std,var_rss23_min,var_rss23_1st_quart,var_rss23_median,var_rss23_3rd_quart,var_rss23_max
0,40.624792,1.476967,37.25,39.2500,40.500,42.0000,45.00,0.358604,0.322605,0.0,...,35.000,36.0000,38.25,0.570583,0.582915,0.0,0.0000,0.430,1.300,1.92
1,42.812812,1.435550,38.00,42.0000,42.500,43.6700,45.67,0.372437,0.289158,0.0,...,33.000,34.5000,38.50,0.571083,0.601010,0.0,0.0000,0.430,1.300,3.11
2,34.227771,4.889576,19.33,30.5000,35.500,37.7500,43.50,3.995729,2.271102,0.0,...,16.670,18.6700,26.00,3.394125,1.792090,0.0,2.1050,3.100,4.425,9.74
3,33.509729,4.850923,12.50,30.5000,34.125,36.7500,45.00,4.450771,2.338685,0.0,...,16.750,18.7500,25.00,3.378479,1.787360,0.0,2.0600,3.085,4.440,8.96
4,34.660583,5.315110,15.00,31.0000,35.000,38.2500,46.75,4.200896,2.480206,0.0,...,16.330,18.5000,24.50,3.244396,1.630983,0.0,2.1200,3.000,4.240,8.99
5,24.562958,3.737514,12.75,23.1875,24.250,26.5000,51.00,0.590833,0.837408,0.0,...,23.750,27.0000,30.00,0.700188,0.693720,0.0,0.4300,0.500,0.870,4.97
6,27.464604,3.583582,0.00,25.5000,28.000,30.0000,42.75,0.449708,0.767197,0.0,...,18.000,20.7500,33.00,1.122125,1.012342,0.0,0.4700,0.830,1.300,6.76
7,44.334729,2.476940,33.33,42.2500,45.000,46.5000,48.00,0.432958,0.370591,0.0,...,14.750,17.7500,23.00,0.933000,0.673609,0.0,0.4700,0.830,1.250,5.02
8,43.174938,1.989052,35.50,42.5000,43.670,44.5000,46.25,0.506583,0.307413,0.0,...,14.670,16.5000,21.25,0.911979,0.666161,0.0,0.4700,0.830,1.220,5.72
9,42.760562,3.398919,32.75,41.3300,44.500,45.3725,47.00,0.486167,0.446511,0.0,...,16.585,18.5650,21.33,0.842271,0.722165,0.0,0.4300,0.710,1.090,5.73


iii. Estimate the standard deviation of each of the time-domain features you
extracted from the data. Then, use Python’s bootstrapped or any other
method to build a 90% bootsrap confidence interval for the standard deviation
of each feature.

In [76]:
train_df.describe().loc['std']

avg_rss12_mean         4.917692
avg_rss12_std          1.758670
avg_rss12_min          8.794295
avg_rss12_1st_quart    5.731647
avg_rss12_median       4.956111
avg_rss12_3rd_quart    4.783645
avg_rss12_max          4.429182
var_rss12_mean         1.600701
var_rss12_std          0.902808
var_rss12_min          0.000000
var_rss12_1st_quart    0.952201
var_rss12_median       1.436960
var_rss12_3rd_quart    2.158416
var_rss12_max          5.147841
avg_rss13_mean         3.863097
avg_rss13_std          0.995959
avg_rss13_min          3.053869
avg_rss13_1st_quart    4.145255
avg_rss13_median       3.845730
avg_rss13_3rd_quart    3.946023
avg_rss13_max          4.759853
var_rss13_mean         1.179861
var_rss13_std          0.473576
var_rss13_min          0.000000
var_rss13_1st_quart    0.842501
var_rss13_median       1.150092
var_rss13_3rd_quart    1.566564
var_rss13_max          2.302408
avg_rss23_mean         5.120426
avg_rss23_std          1.057998
avg_rss23_min          5.368786
avg_rss2

In [89]:
for col in train_df.columns:
    ci = bs.bootstrap(np.array(train_df[col]), stat_func=bs_stats.std, alpha=0.1)
    print(f"90% CI for {col}: {ci}")

90% CI for avg_rss12_mean: 4.881926151830011    (4.3752898311154205, 5.470579315480783)
90% CI for avg_rss12_std: 1.7458793319878128    (1.557576990838869, 1.9587670219396118)
90% CI for avg_rss12_min: 8.730335507644277    (7.49983506144933, 9.98758021292318)
90% CI for avg_rss12_1st_quart: 5.689961410573205    (5.219459869065651, 6.277523514719793)
90% CI for avg_rss12_median: 4.920065980691552    (4.390735724499102, 5.546140225288948)
90% CI for avg_rss12_3rd_quart: 4.748854525045172    (4.005882959579756, 5.596920603507005)
90% CI for avg_rss12_max: 4.396968872496197    (3.435575430233688, 5.548167820583139)
90% CI for var_rss12_mean: 1.5890592339540666    (1.4534419109750547, 1.7888454794439832)
90% CI for var_rss12_std: 0.8962421917100799    (0.8349890343003716, 0.9867742407508776)
90% CI for var_rss12_min: 0.0    (0.0, 0.0)
90% CI for var_rss12_1st_quart: 0.9452754510876189    (0.8505445839524997, 1.0764267247247994)
90% CI for var_rss12_median: 1.4265088556429972    (1.285903774

iv. Use your judgement to select the three most important time-domain features
(one option may be min, mean, and max).

Answer:

To select the best features we need to consider the effect size (standard deviation) in this case and the range of the interval (smaller the range, the better it is). Based on these factors the three important features are:

- avg_rss12_min
- var_rss12_max
- avg_rss12_1st_quart


## 2. ISLR 3.7.4

I collect a set of data (n = 100 observations) containing a single
predictor and a quantitative response. I then fit a linear regression
model to the data, as well as a separate cubic regression, i.e. Y=
β0 + β1X+ β2X2 + β3X3 + ϵ.

(a) Suppose that the true relationship between X and Y is linear,
i.e. Y= β0 + β1X + ϵ. Consider the training residual sum of
squares (RSS) for the linear regression, and also the training
RSS for the cubic regression. Would we expect one to be lower
than the other, would we expect them to be the same, or is there
not enough information to tell? Justify your answer.

Answer: 

Since the linear model fits the data well, we can expect the training RSS for the cubic regression to be lower since it overfits the data

(b) Answer (a) using test rather than training RSS.

Answer:

The test RSS will be lower for linear regression over cubic regression since it generalizes better.

(c) Suppose that the true relationship between X and Y is not linear,
but we don’t know how far it is from linear. Consider the training
RSS for the linear regression, and also the training RSS for the
cubic regression. Would we expect one to be lower than the
other, would we expect them to be the same, or is there not
enough information to tell? Justify your answer.

Answer:

The cubic regression will still have a lower training RSS since it models more complexity from the data points.

(d) Answer (c) using test rather than training RSS.

Answer:

Since we do not know the true nature of the relationship between X and Y, we will need more information to say which model has smaller test RSS.

## Extra Practice ISLR 3.7.3

Suppose we have a data set with five predictors, 
- X1 = GPA, 
- X2 = IQ, 
- X3 = Level (1 for College and 0 for High School), 
- X4 = Interaction between GPA and IQ, and 
- X5 = Interaction between GPA and
Level. 

The response is starting salary after graduation (in thousands
of dollars). Suppose we use least squares to fit the model, and get
β0 = 50,
β1 = 20,
β2 = 0.07,
β3 = 35,
β4 = 0.01,
β5 =−10.


(a) Which answer is correct, and why?

i. For a fixed value of IQ and GPA, high school graduates earn
more, on average, than college graduates.

ii. For a fixed value of IQ and GPA, college graduates earn
more, on average, than high school graduates.

iii. For a fixed value of IQ and GPA, high school graduates earn
more, on average, than college graduates provided that the
GPA is high enough.

iv. For a fixed value of IQ and GPA, college graduates earn
more, on average, than high school graduates provided that
the GPA is high enough.

Answer:

For a fixed value of IQ and GPA, the factors X1, X2 and X4 are going to be fixed for the regression model. The remaining factors are X3 ( \beta_3 = 35) and X5 (\beta_5 = -10). Since there is a negative interaction between GPA and Level, we can say that high school graduates earn more on average provided the GPA is high enough. 

(b) Predict the salary of a college graduate with IQ of 110 and a
GPA of 4.0.

Answer:

The predicted salary would be

S = β0 + β1*X1 + β2*X2 + β3*X3 + β4*X4 + β5*X5
  = 50 + 20*4.0 + 0.07*110 + 35*1 + 0.01*(4.0*110) - 10*(4.0*1)
  = 137.1k USD

(c) True or false: Since the coeﬃcient for the GPA/IQ interaction
term is very small, there is very little evidence of an interaction
eﬀect. Justify your answer.

Answer:

The coefficient does not tell us anything about statistical significance. So we will need p-value to decide the interaction effect.

## Extra Practice ISLR 3.7.5

Consider the fitted values that result from performing linear regression without an intercept. In this setting, the usual formula for the fitted values is:

$$
\hat{y}_i = x_i \hat{\beta}
$$

where

$$
\hat{\beta} = \frac{\sum_{i=1}^n x_i y_i}{\sum_{i=1}^n x_i^2}
$$

Show that the fitted values can be written in the form:

$$
\hat{y}_i = \sum_{j=1}^n a_{ij} y_j
$$

for some constants $a_{ij}$.

What are the values of $a_{ij}$?

Note: We interpret this result by saying that the fitted values from
linear regression are linear combinations of the response values.

Answer:

$$ y_{i} = x_{i}*\beta = x_{i}*\frac{\sum{x_{j}y_{j}}}{\sum{x_{i}^{2}}}$$

$$ = \frac{\sum{x_{i}x_{j}}}{\sum{x_{i}^{2}}} y_{j} $$

$$ = \sum{a_{ij}y_{j}} $$

$$ \implies a_{ij} = \frac{x_{j}\sum{x_{i}}}{\sum{x_{i}^{2}}} $$